In [2]:
!pip install  accelerate sentence-transformers datasets==2.19.1 faiss-cpu pandas numpy  rank-bm25  nltk  rouge-score huggingface_hub hf_transfer tqdm scikit-learn


  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached datasets-2.19.1-py3-none-any.whl.metadata (19 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py) ... done
  Using cached huggingface_hub-1.9.1-py3-none-any.whl.metadata (14 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Usi

In [3]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4070


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA (PyTorch build):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu130
CUDA (PyTorch build): 13.0
CUDA available: False


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:


from huggingface_hub import login

login("")





In [3]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0


In [3]:
pip install bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [6]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 3.8 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 8.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 6.3 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 12.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 2.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 11.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 7.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================
# 1. IMPORTS
# ============================
import os
import gc
import pickle
import numpy as np
import pandas as pd
import torch
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from collections import Counter

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ============================
# 2. CONFIG
# ============================
#for dataset_name, config, split, label in DATASETS:

DATASETS = [
    ("allenai/scifact", "claims", "validation[:50]", "SciFact")
]

MODELS = [
    ("unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "local"),
    ("unsloth/mistral-7b-instruct-v0.3-bnb-4bit", "local")
]

TOP_K = 3

INDEX_PATH = "scifact.index"
DOCS_PATH  = "scifact_texts.pkl"

# ============================
# 3. LOAD RETRIEVER
# ============================
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

if os.path.exists(INDEX_PATH):
    print("Loading cached FAISS index...")
    index = faiss.read_index(INDEX_PATH)
    with open(DOCS_PATH, "rb") as f:
        docs_db, doc_ids = pickle.load(f)
else:
    print("Building SciFact corpus...")
    corpus = load_dataset("allenai/scifact", "corpus", split="train")

    docs_db = []
    doc_ids = []
    
    for doc in corpus:
        text = doc["title"] + ". " + " ".join(doc["abstract"])
        docs_db.append(text)
        doc_ids.append(doc["doc_id"])

    embeddings = embed_model.encode(
        docs_db, batch_size=256, show_progress_bar=True
    ).astype("float32")

    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(DOCS_PATH, "wb") as f:
        pickle.dump((docs_db, doc_ids), f)
        

# move to CPU
embed_model.to("cuda")
torch.cuda.empty_cache()
gc.collect()

def retrieve(query, k=TOP_K):
    q_emb = embed_model.encode([query], device="cuda")

    faiss.normalize_L2(q_emb)   # ← ADD THIS LINE HERE

    distances, indices = index.search(q_emb, k)
    
    docs = [docs_db[i] for i in indices[0]]
    ids  = [doc_ids[i] for i in indices[0]]

    scores = distances[0]
    probs = np.exp(scores - np.max(scores)) / np.sum(np.exp(scores - np.max(scores)))

    return docs, ids, probs
# ============================
# 4. NLI MODEL
# ============================
nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0   # GPU (use -1 if CUDA still unstable)
)
# ============================
# 5. GENERATOR
# ============================
def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        name,
        device_map="auto",
        torch_dtype=torch.float16
    )
    model.eval()
    return tok, model

def generate(tok, model, query, context):
    prompt = f"""
You are a scientific fact checker.

You MUST use ONLY the given context.

Return in this format:

LABEL: SUPPORT / REFUTE / NEUTRAL
EVIDENCE: exact sentence from context

If no supporting sentence exists, write:
EVIDENCE: NONE

Context:
{context}

Claim:
{query}
"""
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

    generated = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(generated, skip_special_tokens=True).strip()

def parse_output(text):
    text = text.upper()

    if "SUPPORT" in text:
        label = "SUPPORT"
    elif "REFUTE" in text or "CONTRADICT" in text:
        label = "REFUTE"
    else:
        label = "NEUTRAL"

    evidence = ""
    if "EVIDENCE:" in text:
        evidence = text.split("EVIDENCE:")[1].strip()

    return label, evidence

# ============================
# 6. TRUE RAG (FIXED)
# ============================


def rag(query, tok, model):
    docs, ids, probs = retrieve(query)

    outputs = []

    for doc, p in zip(docs, probs):
        out = generate(tok, model, query, doc)
        outputs.append((out, p))

    parsed_outputs = [(parse_output(o), p) for o, p in outputs]

    norm_outputs = [(label, p) for (label, _), p in parsed_outputs]
    evidences = [e for (label, e), _ in parsed_outputs]

    # Weighted voting
    score_map = {}
    for label, p in norm_outputs:
        score_map[label] = score_map.get(label, 0) + p

    final = max(score_map, key=score_map.get)

    return docs, ids, probs, [o for o, _ in outputs], final, evidences
    
# ============================
# 7. METRICS
# ============================
def normalize(text):
    text = text.strip().upper()
    if text.startswith("SUPPORT"):
        return "SUPPORT"
    if text.startswith("REFUTE") or text.startswith("CONTRADICT"):
        return "REFUTE"
    return "NEUTRAL"

def compute_em(pred, gt):
    return int(normalize(pred) == normalize(gt))

def compute_span_grounding(docs, evidences):
    grounded = []

    for ev in evidences:
        if ev == "NONE" or len(ev.strip()) == 0:
            grounded.append(0)
            continue

        found = any(ev.lower() in doc.lower() for doc in docs)
        grounded.append(int(found))

    return np.mean(grounded)


# ============================
# 8. hallucination (FIXED)
# ============================
def compute_hallucination_soft(docs, answer, query):
    verdict = normalize(answer)

    # build hypothesis from the MODEL ANSWER
    if verdict == "SUPPORT":
        hypothesis = f"The claim is supported: {query}"
    elif verdict == "REFUTE":
        hypothesis = f"The claim is false: {query}"
    else:
        hypothesis = f"There is insufficient evidence for: {query}"

    inputs = [
        {"text": doc[:512], "text_pair": hypothesis}
        for doc in docs
    ]

    results = nli_model(inputs, truncation=True)

    if verdict == "SUPPORT":
        scores = [r["score"] for r in results if r["label"].upper() == "ENTAILMENT"]

    elif verdict == "REFUTE":
        scores = [r["score"] for r in results if r["label"].upper() == "CONTRADICTION"]

    else:
        return 0.0

    return 1 - max(scores) if scores else 1.0

def compute_hallucination_binary(docs, answer, query):
    verdict = normalize(answer)

    if verdict == "SUPPORT":
        hypothesis = f"The claim is supported: {query}"
    elif verdict == "REFUTE":
        hypothesis = f"The claim is false: {query}"
    else:
        return 0

    inputs = [
        {"text": doc[:512], "text_pair": hypothesis}
        for doc in docs
    ]

    results = nli_model(inputs, truncation=True)

    has_entailment = any(r["label"].upper() == "ENTAILMENT" for r in results)

    return 0 if has_entailment else 1

def compute_final_hallucination(nli_score, span_score):
    return 0.7 * nli_score + 0.3 * (1 - span_score)
    
# ============================
# 9. DATASET PARSER (FIXED)
# ============================
def parse_sample(sample):
    query = sample["claim"]

    label = sample["evidence_label"]

    if label == "SUPPORT":
        gt = "SUPPORT"
    elif label == "CONTRADICT":
        gt = "REFUTE"
    else:
        gt = "NEUTRAL"

    # FIX: handle empty doc_id
    if sample["evidence_doc_id"] == "":
        gold_doc_id = None
    else:
        gold_doc_id = int(sample["evidence_doc_id"])

    return query, gt, gold_doc_id
    
# ============================
# 10. RUN
# ============================
results = []

for dataset_name, config, split, label in DATASETS:
    dataset = load_dataset(dataset_name, config,split=split)

    for model_name, mode in MODELS:
        tok, model = load_model(model_name)

        for i, sample in enumerate(dataset):
            
            query, gt, gold_doc_id = parse_sample(sample)



            docs, ids, probs, answers, pred, evidences = rag(query, tok, model)

            em = compute_em(pred, gt)
            hallucination_nli = compute_hallucination_soft(docs, pred, query)
            hallucination_bin = compute_hallucination_binary(docs, pred, query)
            span_score = compute_span_grounding(docs, evidences)
            hallucination = compute_final_hallucination(hallucination_nli, span_score)
            # DEBUG BLOCK
            print("QUERY:", query)
            print("GT:", gt, "| PRED:", normalize(pred))
            
            # Debug NLI skipped for quick test
            
            print("hal:", hallucination)
            print("Span Score:", span_score)
            print("NLI Score:", hallucination_nli)
            print("-" * 60)

            if gold_doc_id is None:
                recall = None
            else:
                recall = int(gold_doc_id in ids)
            if recall == 0:
                error_type = "retrieval_failure"
            elif em == 0 and hallucination < 0.5:
                error_type = "reasoning_error"
            elif hallucination > 0.7:
                error_type = "hallucination"
            else:
                error_type = "correct"
            
            results.append({
                "model": model_name,

                "gold_doc_id": gold_doc_id,
                "retrieved_doc_ids": ids,
                "retrieval_recall": recall,
                
                "query": query,
                "ground_truth": gt,
                "prediction": normalize(pred),

                "EM": em,
                "hallucination": hallucination,
                "hallucination_bin": hallucination_bin,
                "error_type": error_type,
                
                "top_doc": docs[0],
                "all_docs": docs,
                "doc_probs": probs.tolist(),
                "answers": answers,
                "span_grounding": span_score,
                "evidences": evidences,
            })

            print(f"{i} | EM:{em} | H:{hallucination:.3f}")

        del model
        del tok
        torch.cuda.empty_cache()
        gc.collect()

# ============================
# 11. SAVE
# ============================
df = pd.DataFrame(results)
df.to_csv("final_rag_results.csv", index=False)

summary = df.groupby("model").agg({
    "EM": "mean",
    "hallucination": "mean",
    "retrieval_recall": "mean"
})
print(summary)
summary.to_csv("summary_results.csv")


Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached FAISS index...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

QUERY: Auditory entrainment is strengthened when people see congruent visual and auditory information.
GT: SUPPORT | PRED: SUPPORT
hal: 0.2422150015830994
Span Score: 0.3333333333333333
NLI Score: 0.06030714511871338
------------------------------------------------------------
47 | EM:1 | H:0.242


Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: Auditory entrainment is strengthened when people see congruent visual and auditory information.
GT: SUPPORT | PRED: SUPPORT
hal: 0.2422150015830994
Span Score: 0.3333333333333333
NLI Score: 0.06030714511871338
------------------------------------------------------------
48 | EM:1 | H:0.242


Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: Auditory entrainment is strengthened when people see congruent visual and auditory information.
GT: SUPPORT | PRED: SUPPORT
hal: 0.2422150015830994
Span Score: 0.3333333333333333
NLI Score: 0.06030714511871338
------------------------------------------------------------
49 | EM:1 | H:0.242
                                             EM  hallucination  \
model                                                            
unsloth/Qwen2.5-7B-Instruct-bnb-4bit       0.72       0.484380   
unsloth/mistral-7b-instruct-v0.3-bnb-4bit  0.50       0.522374   

                                           retrieval_recall  
model                                                        
unsloth/Qwen2.5-7B-Instruct-bnb-4bit                    0.9  
unsloth/mistral-7b-instruct-v0.3-bnb-4bit               0.9  
